# 03 - Minh chứng Embedded và hệ thống tích hợp

Notebook phục vụ Q17-Q58 và Q70-Q80: trigger, Quality Gate, threshold, TFLM, parity, packet bảo mật, Gateway, MQTT và các giới hạn khi mất kết nối.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image, Markdown, Code

def find_repo_root(start=Path.cwd()):
    for path in (start, *start.parents):
        if (path / 'Defense_Package').is_dir():
            return path
    raise FileNotFoundError('Không tìm thấy repository root')

ROOT = find_repo_root()
EVIDENCE = ROOT / 'Defense_Package' / '02_Defense_Evidence'

def show_image(relative_path, width=1100):
    path = EVIDENCE / relative_path
    if not path.exists(): raise FileNotFoundError(path)
    display(Image(filename=str(path), width=width))

def show_log(relative_path, lines=80):
    path = EVIDENCE / relative_path
    text = path.read_text(encoding='utf-8', errors='replace').splitlines()
    display(Code('\n'.join(text[-lines:]), language='text'))

print('ROOT =', ROOT)

## 1. VAD, Quality Gate và threshold là ba quyết định khác nhau - Q23, Q70-Q72

1. **VAD**: khi nào kích hoạt cửa sổ.
2. **Quality Gate**: cửa sổ đã thu có đủ điều kiện suy luận không.
3. **Classification threshold**: score của model được đổi thành lớp như thế nào.

In [ ]:
show_image('figures/04_Final_Validation/01_Threshold_Probability_Distribution.png')
show_image('figures/04_Final_Validation/02_Quality_Distributions.png')
for name in ['Classification_Threshold_Final_Metrics.csv', 'Quality_Gate_Test_Metrics.csv', 'VAD_Test_Trigger_Events.csv']:
    print('\n', name)
    display(pd.read_csv(EVIDENCE / 'data' / name).head(20))

## 2. Vì sao suy luận tại Node? - Q21, Q25, Q73, Q80

Edge inference giữ âm thanh cục bộ và vẫn hoạt động khi Gateway/Internet mất. Đổi lại phải chứng minh model vừa bộ nhớ, latency chấp nhận được và pipeline C++ khớp Python.

In [ ]:
show_image('figures/05_Deployment/01_ESP32_Invoke_Logs.png')
show_log('logs/C4_QAT_ESP32_Static.log', 70)
show_log('logs/Raw_to_Inference_Parity.log', 90)

## 3. CHECK, MONITOR và vitals - Q21, Q22, Q76-Q78

CHECK thực hiện một phiên suy luận rồi đo vitals; MONITOR thực hiện ba lần và majority vote. Vitals không được đưa vào model vì chưa có dataset multimodal đồng bộ. Giá trị 0 là sentinel khi timeout, Dashboard phải hiểu là không có dữ liệu.

In [ ]:
show_image('figures/06_System/01_System_Workflow.png')
show_image('figures/06_System/02_Patient_Node.png', width=850)

## 4. ESP-NOW, AES-GCM và Gateway - Q29-Q39, Q74-Q75

ESP-NOW vận chuyển gói nhỏ; AES-GCM bảo mật nội dung và tag; sequence/session hỗ trợ chống replay; ACK/retry xử lý delivery ở tầng ứng dụng. CRC không thay thế xác thực mật mã.

In [ ]:
show_image('figures/06_System/03_Gateway.jpg', width=850)
show_log('logs/Gateway_Test_Result.txt', 120)

## 5. MQTT, Dashboard và giới hạn mất kết nối - Q44-Q56, Q79

Gateway chuyển event thành JSON và publish cho Dashboard. Bằng chứng Wi-Fi/LTE hiện có là code + native/mock test; chưa được gọi là hardware failover nếu chưa có log LTE thật. Hệ thống cũng chưa phải persistent store-and-forward hoàn chỉnh.

In [ ]:
json_path = EVIDENCE / 'data/Dashboard_MQTT_Example.json'
payload = json.loads(json_path.read_text(encoding='utf-8'))
display(payload)
show_image('figures/06_System/04_Patient_Node_PCB.png', width=900)
show_image('figures/06_System/05_Gateway_PCB.png', width=900)

## 6. Câu chốt khi trình bày

> Hệ thống tách trigger, kiểm tra chất lượng và phân loại; suy luận tại Node; truyền kết quả nhỏ qua ESP-NOW có xác thực; Gateway ghép dữ liệu và đưa lên MQTT. Những phần chưa có log hardware thật được nêu là giới hạn, không biến thành kết quả đã kiểm chứng.

Đáp án chi tiết: [Q71](../01_Committee_QnA/Answers/Q71.md), [Q73](../01_Committee_QnA/Answers/Q73.md), [Q74](../01_Committee_QnA/Answers/Q74.md), [Q79](../01_Committee_QnA/Answers/Q79.md), [Q80](../01_Committee_QnA/Answers/Q80.md).